In [0]:
# ============================================================
# MyCredit-IQ | 03_silver_loans_clean.py
# Silver layer: clean, validate, join macro features
# Output: my_living_index.credit.loans_silver
# ============================================================

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = SparkSession.builder.getOrCreate()

In [0]:
# ============================================================
# SECTION 1: LOAD BRONZE TABLES
# ============================================================

loans = spark.table("my_living_index.credit.loans_bronze")
macro = spark.table("my_living_index.credit.macro_bronze")

raw_count = loans.count()
print(f"Loaded: {raw_count:,} rows from loans_bronze")
print(f"Loaded: {macro.count()} rows from macro_bronze")

In [0]:
# ============================================================
# SECTION 2: DATA QUALITY CHECKS
# ============================================================

print("\n── Null Counts per Column ──")
null_counts = loans.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in loans.columns
])
null_counts.show(vertical=True)

print("── Negative / Impossible Values ──")
print(f"  monthly_income <= 0:      {loans.filter(F.col('monthly_income') <= 0).count()}")
print(f"  loan_amount <= 0:         {loans.filter(F.col('loan_amount') <= 0).count()}")
print(f"  monthly_instalment <= 0:  {loans.filter(F.col('monthly_instalment') <= 0).count()}")
print(f"  debt_service_ratio > 1.5: {loans.filter(F.col('debt_service_ratio') > 1.5).count()}")
print(f"  loan_tenure_months <= 0:  {loans.filter(F.col('loan_tenure_months') <= 0).count()}")

print("── Unexpected Categorical Values ──")
loans.groupBy("income_bracket").count().orderBy("income_bracket").show()
loans.groupBy("employment_type").count().orderBy("employment_type").show()
loans.groupBy("loan_type").count().orderBy("loan_type").show()

In [0]:
# ============================================================
# SECTION 3: CLEANING RULES
# ============================================================

loans_clean = (loans
    # Rule 1: Remove rows with null borrower_id (shouldn't exist but defensive)
    .filter(F.col("borrower_id").isNotNull())

    # Rule 2: Remove impossible DSR (> 1.5 = data error, not business risk)
    .filter(F.col("debt_service_ratio") <= 1.5)

    # Rule 3: Remove non-positive income or loan amounts
    .filter(F.col("monthly_income") > 0)
    .filter(F.col("loan_amount") > 0)
    .filter(F.col("monthly_instalment") > 0)

    # Rule 4: Standardise text fields — trim whitespace
    .withColumn("employment_sector",  F.trim(F.col("employment_sector")))
    .withColumn("employment_type",    F.trim(F.col("employment_type")))
    .withColumn("income_bracket",     F.trim(F.col("income_bracket")))
    .withColumn("loan_type",          F.trim(F.col("loan_type")))

    # Rule 5: Cast target to integer explicitly
    .withColumn("is_default", F.col("is_default").cast(IntegerType()))
)

In [0]:
# ============================================================
# SECTION 4: BUSINESS-RULE DERIVED COLUMNS
# ============================================================

loans_flagged = (loans_clean
    # DSR risk band (BNM responsible lending framework reference)
    .withColumn("dsr_band",
                F.when(F.col("debt_service_ratio") <= 0.20, "VERY_LOW")
                .when(F.col("debt_service_ratio") <= 0.40, "LOW")
                .when(F.col("debt_service_ratio") <= 0.60, "MODERATE")
                .when(F.col("debt_service_ratio") <= 0.80, "HIGH")
                .otherwise("VERY_HIGH"))

    # DPD severity (maps to CCRIS classification logic)
    .withColumn("dpd_severity",
        F.when(F.col("days_past_due_history") == 0,   "CLEAN")
         .when(F.col("days_past_due_history") <= 30,  "WATCH")
         .when(F.col("days_past_due_history") <= 60,  "SUBSTANDARD")
         .when(F.col("days_past_due_history") <= 90,  "DOUBTFUL")
         .otherwise("LOSS"))

    # Employment risk class
    .withColumn("employment_risk_class",
        F.when(F.col("employment_type").isin(
            "Gig Worker"), "HIGH")
         .when(F.col("employment_type").isin(
            "Self-Employed"), "ELEVATED")
         .when(F.col("employment_type") == "Contract", "MODERATE")
         .otherwise("STANDARD"))

    # Credit facility load flag
    .withColumn("is_credit_overloaded",
        (F.col("n_credit_facilities") > 5).cast(IntegerType()))

    # High-risk employment binary
    .withColumn("is_informal_employment",
        F.col("employment_type").isin(
            "Self-Employed", "Gig Worker").cast(IntegerType()))
    .withColumn("dsr_dpd_combined_risk",
        F.when((F.col("dsr_band").isin("HIGH", "VERY_HIGH")) & (F.col("dpd_severity").isin("SUBSTANDARD", "DOUBTFUL", "LOSS")), 3)
        .when((F.col("dsr_band").isin("HIGH", "VERY_HIGH")) | (F.col("dpd_severity").isin("SUBSTANDARD", "DOUBTFUL", "LOSS")), 2)
        .when(F.col("dpd_severity") == "WATCH", 1)
        .otherwise(0))

)

In [0]:
# ============================================================
# SECTION 5: JOIN MACRO FEATURES
# Join on origination_ym
# ============================================================

macro_slim = macro.select(
    "origination_ym",
    "opr_rate",
    "gdp_growth_yoy",
    "hh_debt_gdp_ratio",
    "cpi_inflation_yoy",
    "unemployment_rate",
    "economic_environment"
)

loans_silver = loans_flagged.join(
    macro_slim,
    on="origination_ym",
    how="left"
)

# Confirm join worked — no nulls on opr_rate means all months matched
unmatched = loans_silver.filter(F.col("opr_rate").isNull()).count()
print(f"\n── Join Validation ──")
print(f"Unmatched origination_ym (should be 0): {unmatched}")

In [0]:
# ============================================================
# SECTION 6: VALIDATION REPORT
# ============================================================

clean_count = loans_silver.count()

print(f"\n── Silver Layer Summary ──")
print(f"Raw rows:       {raw_count:,}")
print(f"Clean rows:     {clean_count:,}")
print(f"Rows removed:   {raw_count - clean_count:,} ({(raw_count - clean_count)/raw_count:.2%})")

print("\n── DSR Band Distribution ──")
(loans_silver
    .groupBy("dsr_band")
    .agg(
        F.count("*").alias("count"),
        F.round(F.mean("is_default"), 4).alias("default_rate")
    )
    .orderBy("dsr_band")
    .show())

print("── DPD Severity Distribution ──")
(loans_silver
    .groupBy("dpd_severity")
    .agg(
        F.count("*").alias("count"),
        F.round(F.mean("is_default"), 4).alias("default_rate")
    )
    .orderBy("dpd_severity")
    .show())

print("── Default Rate by Economic Environment ──")
(loans_silver
    .groupBy("economic_environment")
    .agg(
        F.count("*").alias("count"),
        F.round(F.mean("is_default"), 4).alias("default_rate"),
        F.round(F.mean("opr_rate"), 3).alias("avg_opr")
    )
    .orderBy("economic_environment")
    .show())

print("── Overall default rate (Silver) ──")
print(f"  {loans_silver.filter('is_default = 1').count() / clean_count:.2%}")

In [0]:
# ============================================================
# SECTION 7: WRITE SILVER TABLE
# ============================================================

(loans_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("my_living_index.credit.loans_silver"))

print(f"\nWritten {clean_count:,} rows → my_living_index.credit.loans_silver")
loans_silver.printSchema()